In [0]:
# Databricks notebook source

# Sprint 5 - Delta Lake Fundamentals

In [0]:
customers = [

    (101, "Neha", "Pune"),

    (102, "Rahul", "Mumbai"),

    (103, "Amit", "Delhi"),

    (104, "Sneha", "Bangalore")

]

columns = [

    "customer_id",

    "customer_name",

    "city"

]

df = spark.createDataFrame(customers, columns)

display(df)

customer_id,customer_name,city
101,Neha,Pune
102,Rahul,Mumbai
103,Amit,Delhi
104,Sneha,Bangalore


In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customers_delta")

In [0]:
display(spark.table("customers_delta"))

customer_id,customer_name,city
101,Neha,Pune
102,Rahul,Mumbai
103,Amit,Delhi
104,Sneha,Bangalore


In [0]:
spark.sql("""
UPDATE customers_delta
SET city = 'Hyderabad'
WHERE customer_id = 101
""")

DataFrame[num_affected_rows: bigint]

In [0]:
display(spark.table("customers_delta"))

customer_id,customer_name,city
101,Neha,Hyderabad
102,Rahul,Mumbai
103,Amit,Delhi
104,Sneha,Bangalore


In [0]:
spark.sql("""
DELETE FROM customers_delta
WHERE customer_id = 104
""")

DataFrame[num_affected_rows: bigint]

In [0]:
display(spark.table("customers_delta"))

customer_id,customer_name,city
101,Neha,Hyderabad
102,Rahul,Mumbai
103,Amit,Delhi


In [0]:
updates = [

    (102, "Rahul", "Chennai"),      # Existing customer -> UPDATE

    (105, "Priya", "Pune"),         # New customer -> INSERT

    (106, "Kiran", "Nagpur")        # New customer -> INSERT

]

updates_df = spark.createDataFrame(
    updates,
    ["customer_id", "customer_name", "city"]
)

display(updates_df)

customer_id,customer_name,city
102,Rahul,Chennai
105,Priya,Pune
106,Kiran,Nagpur


In [0]:
updates_df.createOrReplaceTempView("customer_updates")

In [0]:
spark.sql("""

MERGE INTO customers_delta AS target

USING customer_updates AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN

UPDATE SET *

WHEN NOT MATCHED THEN

INSERT *

""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.table("customers_delta"))

customer_id,customer_name,city
101,Neha,Hyderabad
103,Amit,Delhi
102,Rahul,Chennai
105,Priya,Pune
106,Kiran,Nagpur


In [0]:
display(
    spark.sql("DESCRIBE HISTORY customers_delta")
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-06-30T07:57:32.000Z,71453962787228,dholeneha31284@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3253178975263784),cc169ee4-5d0f-4d81-998e-d37f518ca7cf,0630-075103-igzc18v4-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 5256, p25FileSize -> 1390, numDeletionVectorsRemoved -> 1, minFileSize -> 1390, numAddedFiles -> 1, maxFileSize -> 1390, p75FileSize -> 1390, p50FileSize -> 1390, numAddedBytes -> 1390)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
5,2026-06-30T07:57:30.000Z,71453962787228,dholeneha31284@gmail.com,MERGE,"Map(predicate -> [""(customer_id#12281L = customer_id#12274L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3253178975263784),cc169ee4-5d0f-4d81-998e-d37f518ca7cf,0630-075103-igzc18v4-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 3900, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5024, materializeSourceTimeMs -> 553, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1768, numTargetRowsUpdated -> 1, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2622)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
4,2026-06-30T07:56:14.000Z,71453962787228,dholeneha31284@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3253178975263784),da0a2174-566e-4354-9423-d0128cd9f178,0630-075103-igzc18v4-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1382, p25FileSize -> 1356, numDeletionVectorsRemoved -> 1, minFileSize -> 1356, numAddedFiles -> 1, maxFileSize -> 1356, p75FileSize -> 1356, p50FileSize -> 1356, numAddedBytes -> 1356)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
3,2026-06-30T07:56:11.000Z,71453962787228,dholeneha31284@gmail.com,DELETE,"Map(predicate -> [""(customer_id#11958L = 104)""])",null,List(3253178975263784),da0a2174-566e-4354-9423-d0128cd9f178,0630-075103-igzc18v4-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2055, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1311, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 741)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
2,2026-06-30T07:54:59.000Z,71453962787228,dholeneha31284@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3253178975263784),52841ebe-0d6f-4217-b048-708b3708b6c5,0630-075103-igzc18v4-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2684, p25FileSize -> 1382, numDeletionVectorsRemoved -> 1, minFileSize -> 1382, numAddedFiles -> 1, maxFileSize -> 1382, p75FileSize -> 1382, p50FileSize -> 1382, numAddedBytes -> 1382)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-06-30T07:54:56.000Z,71453962787228,dholeneha31284@gmail.com,UPDATE,"Map(predicate -> [""(customer_id#11519L = 101)""])",null,List(3253178975263784),52841ebe-0d6f-4217-b048-708b3708b6c5,0630-075103-igzc18v4-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdde

In [0]:
display(
    spark.read
         .option("versionAsOf", 0)
         .table("customers_delta")
)

customer_id,customer_name,city
101,Neha,Pune
102,Rahul,Mumbai
103,Amit,Delhi
104,Sneha,Bangalore


In [0]:
display(spark.table("customers_delta"))

customer_id,customer_name,city
101,Neha,Hyderabad
103,Amit,Delhi
102,Rahul,Chennai
105,Priya,Pune
106,Kiran,Nagpur


In [0]:
spark.sql("OPTIMIZE customers_delta")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
display(
    spark.sql("OPTIMIZE customers_delta")
)

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1782806480230, 1782806480890, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
display(
    spark.sql("""
    VACUUM customers_delta RETAIN 168 HOURS
    """)
)

path
""


In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY customers_delta
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-06-30T08:02:21.000Z,71453962787228,dholeneha31284@gmail.com,VACUUM END,Map(status -> COMPLETED),null,List(3253178975263784),71452d70-d024-4882-b2f6-9d3dc975f841,0630-075103-igzc18v4-v2n,7,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
7,2026-06-30T08:02:20.000Z,71453962787228,dholeneha31284@gmail.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(3253178975263784),71452d70-d024-4882-b2f6-9d3dc975f841,0630-075103-igzc18v4-v2n,6,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
6,2026-06-30T07:57:32.000Z,71453962787228,dholeneha31284@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3253178975263784),cc169ee4-5d0f-4d81-998e-d37f518ca7cf,0630-075103-igzc18v4-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 5256, p25FileSize -> 1390, numDeletionVectorsRemoved -> 1, minFileSize -> 1390, numAddedFiles -> 1, maxFileSize -> 1390, p75FileSize -> 1390, p50FileSize -> 1390, numAddedBytes -> 1390)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
5,2026-06-30T07:57:30.000Z,71453962787228,dholeneha31284@gmail.com,MERGE,"Map(predicate -> [""(customer_id#12281L = customer_id#12274L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3253178975263784),cc169ee4-5d0f-4d81-998e-d37f518ca7cf,0630-075103-igzc18v4-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 3900, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5024, materializeSourceTimeMs -> 553, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1768, numTargetRowsUpdated -> 1, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2622)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
4,2026-06-30T07:56:14.000Z,71453962787228,dholeneha31284@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3253178975263784),da0a2174-566e-4354-9423-d0128cd9f178,0630-075103-igzc18v4-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1382, p25FileSize -> 1356, numDeletionVectorsRemoved -> 1, minFileSize -> 1356, numAddedFiles -> 1, maxFileSize -> 1356, p75FileSize -> 1356, p50FileSize -> 1356, numAddedBytes -> 1356)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
3,2026-06-30T07:56:11.000Z,71453962787228,dholeneha31284@gmail.com,DELETE,"Map(predicate -> [""(customer_id#11958L = 104)""])",null,List(3253178975263784),da0a2174-566e-4354-9423-d0128cd9f178,0630-075103-igzc18v4-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2055, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1311, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 741)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
2,2026-06-30T07:54:59.000Z,71453962787228,dholeneha31284@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3253178975263784),5

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE customers_cdf (

    customer_id INT,

    customer_name STRING,

    city STRING

)

TBLPROPERTIES (

    delta.enableChangeDataFeed = true

)
""")

DataFrame[]

In [0]:
spark.sql("""
INSERT INTO customers_cdf VALUES

(101,'Neha','Pune'),

(102,'Rahul','Mumbai')

""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table("customers_cdf")
)

customer_id,customer_name,city
101,Neha,Pune
102,Rahul,Mumbai


In [0]:
spark.sql("""
UPDATE customers_cdf

SET city='Hyderabad'

WHERE customer_id=101

""")

DataFrame[num_affected_rows: bigint]

In [0]:
display(
    spark.table("customers_cdf")
)

customer_id,customer_name,city
102,Rahul,Mumbai
101,Neha,Hyderabad


In [0]:
display(
    spark.sql("""
    SELECT *
    FROM table_changes('customers_cdf', 0)
    """)
)

customer_id,customer_name,city,_change_type,_commit_version,_commit_timestamp
101,Neha,Pune,update_preimage,2,2026-06-30T08:04:14.000Z
101,Neha,Hyderabad,update_postimage,2,2026-06-30T08:04:14.000Z
101,Neha,Pune,insert,1,2026-06-30T08:03:26.000Z
102,Rahul,Mumbai,insert,1,2026-06-30T08:03:26.000Z
